# training-step-cycle — ex4: modify the cycle for N-step gradient accumulation

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `training-step-cycle`. Running the final beacon cell reports progress against the `PyTorch: Training step cycle` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Training step cycle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`training-step-cycle`** (exercise 4). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "training-step-cycle"
DD_SUBTOPIC = "PyTorch: Training step cycle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Training-step cycle — quick refresher

The canonical step is forward → loss → backward → step → zero_grad. `backward()` ACCUMULATES into `param.grad`, so failing to call `zero_grad()` makes consecutive batches add their grads. **Gradient accumulation** exploits that behaviour deliberately: you call `backward()` on `N` micro-batches and only fire `step()` + `zero_grad()` after the N-th, simulating an N×-larger effective batch.

### Exercise 4 — modify the cycle for N-step gradient accumulation

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Modify the canonical 5-call training step into an N-micro-batch gradient-accumulation variant that defers `step()` and `zero_grad()` until every Nth micro-batch.
> Keywords: gradient-accumulation, micro-batch, effective-batch-size, modify-the-cycle
> ```

**KCs targeted:** `training-step-five-call-order`, `training-step-zero-grad-resets-accumulation`

ex1–ex3 explored the *correct order* and *ordering bugs* of the canonical 5-call cycle. This one *deliberately modifies* the cycle to exploit grad accumulation — the standard trick for fitting a large effective batch on a small device.

Implement `ex4_accumulate(model, x_batches, y_batches, optimizer, loss_fn, accum_steps)`:

1. `x_batches` and `y_batches` are length-`M` lists of micro-batch tensors, where `M % accum_steps == 0`.
2. For each micro-batch `i`:
   - forward + loss + backward (accumulate grads into `param.grad`)
   - only call `optimizer.step()` + `optimizer.zero_grad()` when `(i + 1) % accum_steps == 0`.
3. Divide each micro-batch loss by `accum_steps` BEFORE backward so the accumulated grad equals the average of the micro-batch grads (same magnitude as a single big batch).
4. Return a list of the loss values *before* the division (one per micro-batch), so the caller can plot loss per micro-step.

The test verifies that the parameter trajectory after `accum_steps=2` on `M=4` micro-batches matches a single-batch baseline run on the concatenated data — i.e. that the modification is mathematically equivalent.

In [ ]:
def ex4_accumulate(model, x_batches, y_batches, optimizer,
                   loss_fn, accum_steps: int) -> list:
    """Gradient-accumulated training loop. Returns per-micro-batch losses."""
    raise NotImplementedError()


def _test_ex4():
    from torch import nn

    def make_model():
        rng = t.Generator().manual_seed(123)
        m = nn.Linear(3, 1, bias=False)
        with t.no_grad():
            m.weight.copy_(t.randn(1, 3, generator=rng))
        return m

    rng = t.Generator().manual_seed(0)
    X = t.randn(8, 3, generator=rng)
    y = t.randn(8, 1, generator=rng)
    x_micro = list(X.split(2))   # 4 micro-batches of size 2
    y_micro = list(y.split(2))

    # Baseline: ONE big step on the concatenated data, lr / accum.
    baseline = make_model()
    opt_b = t.optim.SGD(baseline.parameters(), lr=0.1)
    loss_fn = nn.MSELoss()
    opt_b.zero_grad()
    loss_b = loss_fn(baseline(X), y)
    loss_b.backward()
    opt_b.step()
    ref_w = baseline.weight.detach().clone()

    # Test: 4-micro-batch run with accum_steps=4.
    model = make_model()
    opt = t.optim.SGD(model.parameters(), lr=0.1)
    # Pre-zero so the first backward starts clean (typical usage).
    opt.zero_grad()
    losses = ex4_accumulate(model, x_micro, y_micro, opt, loss_fn, accum_steps=4)
    assert isinstance(losses, list), f'expected list, got {type(losses).__name__}'
    assert len(losses) == 4, f'expected 4 losses, got {len(losses)}'
    got_w = model.weight.detach().clone()
    assert t.allclose(got_w, ref_w, atol=1e-5), (
        f'parameter mismatch after accumulation:\n'
        f'  got:      {got_w}\n'
        f'  baseline: {ref_w}\n'
        'gradient accumulation must equal a single big-batch step.'
    )

    # Sanity: accum_steps=1 == standard per-micro-batch SGD trajectory.
    rng2 = t.Generator().manual_seed(0)
    X2 = t.randn(6, 3, generator=rng2)
    y2 = t.randn(6, 1, generator=rng2)
    x_m2 = list(X2.split(2)); y_m2 = list(y2.split(2))
    model_a = make_model(); opt_a = t.optim.SGD(model_a.parameters(), lr=0.05)
    opt_a.zero_grad()
    ex4_accumulate(model_a, x_m2, y_m2, opt_a, loss_fn, accum_steps=1)
    model_b = make_model(); opt_b2 = t.optim.SGD(model_b.parameters(), lr=0.05)
    for xi, yi in zip(x_m2, y_m2):
        opt_b2.zero_grad()
        L = loss_fn(model_b(xi), yi)
        L.backward()
        opt_b2.step()
    assert t.allclose(model_a.weight, model_b.weight, atol=1e-5), 'accum_steps=1 must match per-batch SGD'

    # --- Visualization: loss curve over micro-batches (real run) ---
    rng3 = t.Generator().manual_seed(5)
    n_micro = 16
    Xv = t.randn(n_micro * 4, 3, generator=rng3)
    true_w = t.tensor([[1.0, -2.0, 0.5]])
    yv = Xv @ true_w.T + 0.1 * t.randn(n_micro * 4, 1, generator=rng3)
    model_v = nn.Linear(3, 1, bias=False)
    opt_v = t.optim.SGD(model_v.parameters(), lr=0.05)
    opt_v.zero_grad()
    loss_curve = ex4_accumulate(model_v, list(Xv.split(4)), list(yv.split(4)),
                                opt_v, loss_fn, accum_steps=4)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(loss_curve, marker='o')
    for k in range(4, n_micro + 1, 4):
        ax.axvline(k - 1, color='red', alpha=0.3, linestyle='--')
    ax.set_xlabel('micro-batch index')
    ax.set_ylabel('MSE loss (pre-division)')
    ax.set_title('ex4 grad-accum: red dashes = optimizer step boundaries')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_accumulate(model, x_batches, y_batches, optimizer,
                   loss_fn, accum_steps: int) -> list:
    losses = []
    for i, (xb, yb) in enumerate(zip(x_batches, y_batches)):
        # 1. forward
        pred = model(xb)
        # 2. loss
        loss = loss_fn(pred, yb)
        losses.append(loss.item())
        # 3. backward (scaled so accumulated grad == averaged grad)
        (loss / accum_steps).backward()
        # 4. step + 5. zero_grad — only on the Nth micro-batch
        if (i + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
    return losses
```

**Why divide loss by `accum_steps`.** `MSELoss` already averages WITHIN one micro-batch, so accumulating N micro-batches without rescaling gives a grad that's `N×` too big. Dividing the loss by `N` before each backward makes the accumulated grad equal to the grad you'd get from one big batch of size `N × B_micro`.

**Why the test pre-zeroes once outside the helper.** The helper only zeroes AFTER the Nth backward. Without an initial `zero_grad()` the very first backward would accumulate into whatever was already in `param.grad` (zero for a fresh model, but garbage if you reuse the model across calls).

**Step-before-backward bugs (ex3) get sneakier here.** With accumulation, calling `step()` every micro-batch (not every Nth) trains correctly but with `accum_steps×` effective LR — easy to miss because the loss still goes down. The mathematical-equivalence assertion catches it.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex4',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()